# 과제 10 · 06 · 조회 Agent 를 돌리고 실제로 무엇을 불렀는지 감사하기

## 현업 시나리오
이력 담당 Agent 는 먼저 판정 기준을 확인하고, 그 결과를 본 뒤 **예측 유형과 기준이 성립한 유형 모두**의 이력을 찾아야 합니다. 예측은 OSF 인데 기준은 TWF 가 성립하는 카드에서 OSF 이력만 찾고 끝내면 TWF 이력을 놓칩니다. 실제 모델(gpt-4.1-mini)은 순서는 지켰지만 필요한 유형을 빠뜨린 적이 있습니다.

## 학습 목표
- `create_agent` 에 Tool 을 쥐여 주고 `ModelCallLimitMiddleware` 로 판단 횟수에 상한을 건다.
- Agent 가 남긴 메시지 기록에서 턴마다 무엇을 불렀는지 읽는다.
- 필요한 조회 목록과 대조해 의존 순서 위반과 빠진 유형을 찾는다.

## 직접 조립
완성된 App을 가져오지 않습니다. 아래에서 작은 fixture와 핵심 함수를 직접 만듭니다.

### 1단계 · Agent 에 Tool 을 쥐여 주고 돌리기

In [ ]:
# 이력 담당이 쓸 Tool 둘입니다. 판정 기준을 계산하는 것과 같은 유형의 이력을 찾는 것.
# 어느 유형의 이력을 찾을지는 판정 결과를 봐야 압니다. 그래서 둘 사이에 의존이 있습니다.
import json
from typing import Literal
from langchain_core.tools import tool

PRACTICE_HISTORY = {"OSF": ["MR-0201"], "TWF": ["MR-0104", "MR-0188"], "HDF": ["MR-0102"]}

@tool
def check_criteria(tool_wear_min: float, torque_nm: float) -> str:
    """매뉴얼 판정 기준을 센서 값에 적용해 기준이 성립한 유형을 돌려준다."""
    met = (["TWF"] if tool_wear_min >= 200 else []) + (["OSF"] if tool_wear_min * torque_nm > 11000 else [])
    return json.dumps({"met": met})

@tool
def same_type_history(failure_type: Literal["TWF", "HDF", "PWF", "OSF"], before: str) -> str:
    """정비 이력에서 이 유형으로 수리한 기록을 찾는다. before 시각보다 앞선 기록만 돌려준다."""
    return json.dumps({"records": PRACTICE_HISTORY.get(failure_type, [])})

# 루프는 create_agent 가 돌리고, 판단 횟수 상한은 ModelCallLimitMiddleware 가 셉니다.
# 관찰 포인트: 대본은 "무엇을 부를지" 만 정합니다. Tool 은 진짜로 실행되어 ToolMessage 를 남깁니다.
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langchain_core.messages import AIMessage

# 대본대로 Tool 을 부르는 모델입니다. LangChain 이 주는 가짜 모델 중
# bind_tools 를 구현한 것이 없어 그 한 가지만 덮습니다.
from langchain_core.language_models import GenericFakeChatModel

class PracticeScriptedModel(GenericFakeChatModel):
    """무엇을 부를지는 대본이 이미 정했으므로 Tool 목록을 보지 않습니다."""
    def bind_tools(self, tools, **kwargs):
        return self

PRACTICE_EVENT_AT = "2025-09-14T02:30:00+09:00"
PRACTICE_PREDICTED = "OSF"
PRACTICE_BUDGET = 4

def practice_run(script, budget=PRACTICE_BUDGET):
    agent = create_agent(PracticeScriptedModel(messages=iter(script)), [check_criteria, same_type_history],
                         middleware=[ModelCallLimitMiddleware(run_limit=budget, exit_behavior="end")])
    return agent.invoke({"messages": [("user", "EVT-2025-0106 의 근거를 모아 주세요")]})["messages"]

practice_script = [
    AIMessage(content="", tool_calls=[{"name": "check_criteria", "args": {"tool_wear_min": 214, "torque_nm": 48.0}, "id": "c1"}]),
    AIMessage(content="", tool_calls=[
        {"name": "same_type_history", "args": {"failure_type": "OSF", "before": PRACTICE_EVENT_AT}, "id": "h1"},
        {"name": "same_type_history", "args": {"failure_type": "TWF", "before": PRACTICE_EVENT_AT}, "id": "h2"}]),
    AIMessage(content="판정과 이력 조회를 마쳤습니다."),
]
practice_agent = practice_run(practice_script)
for practice_message in practice_agent:
    practice_calls = [c["name"] + str(c["args"].get("failure_type", "")) for c in getattr(practice_message, "tool_calls", [])]
    print(f"  {type(practice_message).__name__:12} {practice_calls or str(practice_message.content)[:50]}")

**결과 해설** — Agent 는 세 번 판단했습니다. 첫 턴에 판정 기준을 부르고, 결과(TWF 성립)를 본 둘째 턴에 OSF 와 TWF 이력을 함께 부르고, 셋째 턴에 끝냈습니다. ToolMessage 가 실제 실행 결과입니다.

### 2단계 · 기록에서 감사하기

In [ ]:
# 대본을 믿지 않고 기록에서 읽습니다. AIMessage 의 tool_calls 마다 몇 번째 턴인지 셉니다.
# 필요한 유형은 "예측 유형 + 판정 기준이 성립한 유형" 입니다. 판정 결과는 ToolMessage 에서 꺼냅니다.
# 관찰 포인트: 의존 순서(이력은 판정 다음 턴)와 빠진 유형을 둘 다 봅니다. 둘은 다른 실패입니다.
from langchain_core.messages import ToolMessage

def practice_audit(messages):
    turn, turns, called, met = 0, {}, [], []
    for message in messages:
        if isinstance(message, AIMessage) and message.tool_calls:
            turn += 1
            for call in message.tool_calls:
                turns.setdefault(call["name"], []).append(turn)
                if call["name"] == "same_type_history":
                    called.append(call["args"]["failure_type"])
        if isinstance(message, ToolMessage) and '"met"' in str(message.content):
            met = json.loads(message.content)["met"]
    needed = list(dict.fromkeys([PRACTICE_PREDICTED, *met]))
    first_criteria = min(turns.get("check_criteria", [99]))
    return {"needed": needed, "called": called,
            "missing": [t for t in needed if t not in called],
            "dependency_respected": bool(turns.get("same_type_history")) and
                                    all(t > first_criteria for t in turns["same_type_history"])}

practice_ok = practice_audit(practice_agent)
print(practice_ok)

**결과 해설** — 필요한 유형은 OSF(예측)와 TWF(기준 성립)이고, 둘 다 불렀으며 둘 다 판정 다음 턴이었습니다. 이 감사는 대본이 무엇이었는지 모릅니다. 메시지 기록만 봅니다. 그래서 live 에서 실제 모델이 돌 때도 같은 함수로 확인할 수 있습니다.

### 3단계 · 빠뜨리면, 그리고 상한에 걸리면

In [ ]:
# 실패 둘을 만듭니다. 첫째는 판정을 보기 전에 예측 유형 이력만 부르고 끝내는 대본입니다.
# 둘째는 정상 대본이지만 판단 횟수 상한을 1 로 줄인 경우입니다.
# 관찰 포인트: 둘 다 오류 없이 끝납니다. 감사가 없으면 TWF 이력이 빠진 것을 아무도 모릅니다.
practice_skipping = practice_run([
    AIMessage(content="", tool_calls=[
        {"name": "check_criteria", "args": {"tool_wear_min": 214, "torque_nm": 48.0}, "id": "c1"},
        {"name": "same_type_history", "args": {"failure_type": "OSF", "before": PRACTICE_EVENT_AT}, "id": "h1"}]),
    AIMessage(content="조회를 마쳤습니다."),
])
practice_skip_audit = practice_audit(practice_skipping)
practice_cut = practice_run(practice_script, budget=1)
practice_cut_audit = practice_audit(practice_cut)
print("빠뜨린 대본:", practice_skip_audit)
print("상한 1     :", practice_cut_audit, "· 마지막 메시지:", practice_cut[-1].content[:40])

assert practice_ok == {"needed": ["OSF", "TWF"], "called": ["OSF", "TWF"], "missing": [], "dependency_respected": True}
assert practice_skip_audit["missing"] == ["TWF"] and practice_skip_audit["dependency_respected"] is False
assert practice_cut_audit["missing"] == ["OSF", "TWF"], "상한 1 인데 이력 조회까지 갔습니다"
assert "limit" in practice_cut[-1].content

**결과 해설** — 빠뜨린 대본은 판정과 같은 턴에 OSF 이력을 불러 의존 순서를 어겼고(`dependency_respected=False`), TWF 를 빠뜨렸습니다. 상한 1 은 판정만 하고 멈춰 이력을 하나도 찾지 못했고, 마지막 메시지가 상한에 걸렸다고 말합니다. **둘 다 예외 없이 끝났습니다.** 그래서 앱은 감사 결과로 빠진 유형을 코드가 채웁니다.

## 직접 해 볼 과제
`practice_try_budget` 을 2, 3 으로 바꿔 정상 대본을 다시 돌려 보세요. 몇부터 빠진 유형이 없어집니까? 정상 경로에 판단이 몇 번 필요한지와 앱이 상한을 4 로 둔 이유를 연결해 보세요.

In [ ]:
# 상한만 바꿔 같은 대본을 돌립니다. 판단 횟수 = AIMessage 를 만든 횟수입니다.
# 상한이 정상 경로보다 작으면 일을 다 못 하고, 너무 크면 헛도는 모델을 오래 둡니다.
practice_try_budget = 2
practice_try = practice_run(practice_script, budget=practice_try_budget)
print("상한", practice_try_budget, "→", practice_audit(practice_try), "· 끝:", practice_try[-1].content[:30])

## 중간 결과
정상 대본의 턴별 호출, 감사 결과(필요한 유형·부른 유형·빠진 유형·의존 순서), 그리고 빠뜨린 대본과 상한 1 에서 감사가 무엇을 잡는지 확인합니다.

## 실패 경계
감사는 "불렀는가"를 볼 뿐 "제대로 돌았는가"는 따로 봐야 합니다. MCP 를 거치면 실패한 호출이 정상 결과 글자로 돌아와, 부른 것만 세면 실패를 성공으로 셉니다(다음 Notebook). 또 상한에 걸린 Agent 는 예외 없이 끝나므로, 응답이 상한 소진을 밝혀야 사람이 압니다.

## 실제 app 연결
과제 10 App 의 `lookup.py::fixture_plan` 이 같은 순서의 대본(fixture)이고, live 에서는 실제 모델이 스스로 고릅니다. `lookup.py::read_run` 이 기록에서 의존 순서와 빠진 유형을 읽고, `lookup.py::STEP_BUDGET` 이 상한 4 입니다. Agent 조립은 `shared/tools/agent.py::build_agent` 가 맡습니다. 빠진 유형은 `review.py::collect_history` 가 코드로 채우고 응답의 `filled_history_types` 에 남깁니다.

## 다음 Notebook 연결
다음 `07_mcp_history_server.ipynb` 에서는 같은 Tool 을 별도 Process 의 MCP 서버로 엽니다.